# Exploratory Data Analysis — Smart Waste Classification

Covers: class distribution, image dimension/quality checks, duplicate detection, and a sample image grid.
Run `preprocessing/preprocess.py --remap` first to generate `data/consolidated/`.

In [ ]:
from pathlib import Path
import random
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

DEST = Path("../data/consolidated")
FIG_DIR = Path("../results/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

## 1. Class Distribution

In [ ]:
counts = {d.name: len(list(d.glob("*.*"))) for d in sorted(DEST.iterdir())}

plt.figure(figsize=(8, 5))
plt.bar(counts.keys(), counts.values(), color="teal")
plt.title("Class Distribution — Smart Waste Dataset")
plt.ylabel("Image count")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(FIG_DIR / "class_distribution.png")
plt.show()

counts

## 2. Image Dimensions & Corruption Check

In [ ]:
records, corrupted = [], []
for cls_dir in DEST.iterdir():
    for img_path in cls_dir.glob("*.*"):
        try:
            with Image.open(img_path) as im:
                records.append({"class": cls_dir.name, "width": im.width, "height": im.height, "mode": im.mode})
        except Exception as e:
            corrupted.append((str(img_path), str(e)))

df = pd.DataFrame(records)
print(f"Corrupted/unreadable files: {len(corrupted)}")
df.groupby("class")[["width", "height"]].describe()

## 3. Duplicate Detection (perceptual hash)

In [ ]:
import imagehash
from collections import defaultdict

hashes = defaultdict(list)
for cls_dir in DEST.iterdir():
    for img_path in cls_dir.glob("*.*"):
        h = imagehash.average_hash(Image.open(img_path))
        hashes[str(h)].append(img_path)

dupes = {h: paths for h, paths in hashes.items() if len(paths) > 1}
print(f"Duplicate groups found: {len(dupes)}")

## 4. Sample Image Grid

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for i, cls_dir in enumerate(sorted(DEST.iterdir())):
    imgs = list(cls_dir.glob("*.*"))
    for j in range(2):
        img = Image.open(random.choice(imgs))
        axes[j][i].imshow(img)
        axes[j][i].axis("off")
        if j == 0:
            axes[j][i].set_title(cls_dir.name, fontsize=10)
plt.tight_layout()
plt.savefig(FIG_DIR / "sample_grid.png")
plt.show()